# Positive Expected Profit and Loss Risk: A Two-Dice Simulation Study

Research question:
How does session length affect the probability of losing money in a positive-expectation dice game, and how accurately do simulation and normal approximation estimate that probability?

## Section 1: Introduction
- Rules: Pay $4, roll two fair six-sided dice, and receive the higher number in dollars.
- Questions: How does session length affect the chance of finishing with a loss, and how accurately can we calculate or estimate it?
- Assumptions: Dice and rounds are independent; the entry fee stays fixed; there are no additional costs.
- Definition: A session is a fixed number of games. A loss means its total profit is below 0; breaking even does not count.

## Section 2: Theoretical Analysis
Let M be the higher of the two dice.

There are k² outcomes where both dice are at most k, and (k − 1)² outcomes where both dice are below k. Therefore:

$$
P(M=k)=\frac{k^2-(k-1)^2}{36}=\frac{2k-1}{36}, \quad k=1, \ldots,6.
$$

The expected output is:

$$
E[M]=\sum_{k=1}^{6}k\frac{2k-1}{36}
=\frac{161}{36}\approx 4.4722.
$$

Subtracting the $4 entry fee gives:

$$
E[\text{profit}]=\frac{161}{36}-4
=\frac{17}{36}\approx \$0.4722.
$$


## Section 3: Monte Carlo Simulation

Monte Carlo simulation uses repeated random trials to estimate quantities
such as average profit and loss probability. I will start my simulating one round of the game.

In [111]:
import random

random.seed(1)

die_1 = random.randint(1, 6)
die_2 = random.randint(1, 6)

payout = max(die_1, die_2)
profit = payout - 4

print("Dice:", die_1, die_2)
print("Payout:", payout)
print("Profit:", profit)

Dice: 2 5
Payout: 5
Profit: 1


In [112]:
def one_round_simulation() -> int:
    die_1 = random.randint(1, 6)
    die_2 = random.randint(1, 6)
    
    return max(die_1, die_2) - 4

number_of_games = 10_000
total_profit = 0

random.seed(1)

for game in range(number_of_games):
    total_profit += one_round_simulation()

average_profit = total_profit / number_of_games

print("Average profit per game:", average_profit)
print("Theoretical expected profit:", 17 / 36)


Average profit per game: 0.4702
Theoretical expected profit: 0.4722222222222222


With seed 1, the simulation of 10,000 gaems produced an average profit
of `$0.4702` per game, compared with the theoretical expected profit of
approximately `$0.4722`. The differences reflects random sampling variation.

In [113]:
def simulate_session(games_per_sessions: int) -> int:
    session_profit = 0
    
    for game in range(games_per_sessions):
        session_profit += one_round_simulation()
        
    return session_profit

In [114]:
def simulate_loss_probability(games_per_sessions, number_of_sessions):
    losing_sessions = 0
    
    for session in range(number_of_sessions):
        session_profit = simulate_session(games_per_sessions)
        
        if session_profit < 0:
            losing_sessions += 1
    return losing_sessions / number_of_sessions

In [115]:
random.seed(1)
print(simulate_loss_probability(10, 100_000))

0.12427


I simulated 100,000 sessions containing 10 games each, using seed 1.
A session counted as a loss only when its total profit was negative.

The estimated loss probability was 0.12427, or 12.427%. This shows that
a positive expected profit per game does not guarantee a profitable 10-game
session.

## Section 4: Exact Probability Calculation

Simulation gives an estimate that varies with the random sample. 
To check it, I calculate the exact loss probability using dynamic programming.

For each round, I track how many dice sequences produce each possible 
total profit. After the final round, I add the counts for negative totals and divided by 36 raised to the number of games.

In [116]:
def exact_loss_probability(number_of_games):
    counts = {0: 1}
    outcomes = {-3: 1, -2: 3, -1: 5, 0: 7, 1: 9, 2: 11}
    
    for game in range(number_of_games):
        new_counts = {}
        
        for total_profit, count in counts.items():
            for profit, outcome_count in outcomes.items():
                new_total = total_profit + profit
                new_counts[new_total] = (
                    new_counts.get(new_total, 0) + count * outcome_count
                )
            counts = new_counts
    
    losing_sequences = sum(
        count for total_profit, count in counts.items()
        if total_profit < 0
    )
    
    total_sequences = 36 ** number_of_games
    return losing_sequences / total_sequences

In [117]:
session_lengths = [1, 5, 10, 20, 50, 100]
simulation_results = []
exact_results = []

random.seed(1)

for n in session_lengths:
    simulated = simulate_loss_probability(n, 100_000)
    exact = exact_loss_probability(n)
    
    simulation_results.append(simulated)
    exact_results.append(exact)
    
    print(
        f"{n} games: simulation = {simulated:.6f}, "
        f"exact = {exact:.6f}"
    )

1 games: simulation = 0.249680, exact = 0.250000
5 games: simulation = 0.182730, exact = 0.182144
10 games: simulation = 0.122930, exact = 0.123189
20 games: simulation = 0.061750, exact = 0.060521
50 games: simulation = 0.008610, exact = 0.008982
100 games: simulation = 0.000400, exact = 0.000473


### Simulation versus exact probability

Across the session lengths tested, simulated loss probabilities were
clost to the exact values. Differences reflect random sampling variation.

The exact probability of finishing with a loss fell from 25% for one game
to approximately 0.0473% for 100 games. Although expected profit is positive,
a session can still finish with a loss.

## Section 5: Normal Approximation

The Central Limit Theorem (CLT) suggests that the total profit over many independent gaems can be approximated by a normal distribution.

I compare this approximation with the exact loss probability to assess its accuracy.

In [118]:
from math import sqrt

outcomes = {-3: 1, -2: 3, -1: 5, 0: 7, 1: 9, 2: 11}

mean_profit = sum(profit * count / 36
                  for profit, count in outcomes.items()
)

expected_squared_profit = sum(
    profit ** 2 * count / 36
    for profit, count in outcomes.items()
)

variance_profit = expected_squared_profit - mean_profit ** 2
std_profit = sqrt(variance_profit)

print("Mean profit:", mean_profit)
print("Expected squared profit:", expected_squared_profit)
print("Variance:", variance_profit)
print("Standard deviation:", std_profit)

Mean profit: 0.47222222222222227
Expected squared profit: 2.1944444444444446
Variance: 1.9714506172839508
Standard deviation: 1.4040835506777902


In [119]:
n = 10

session_mean = n * mean_profit
session_std = sqrt(n) * std_profit

print("Session mean:", session_mean)
print("Session standard deviation:", session_std)

Session mean: 4.722222222222222
Session standard deviation: 4.440102045318273


In [120]:
z_score = (-0.5 - session_mean) / session_std
print("Z-score:", z_score)

Z-score: -1.1761491445289263


In [121]:
from statistics import NormalDist

clt_probability = NormalDist().cdf(z_score)

print(f"CLT approximation: {clt_probability:.8f}")
print(f"Exact probability: {exact_loss_probability(10):.8f}")

CLT approximation: 0.11976764
Exact probability: 0.12318872


In [122]:
def clt_loss_probability(number_of_games):
    session_mean = number_of_games * mean_profit
    session_std = sqrt(number_of_games) * std_profit

    z_score = (-0.5 - session_mean) / session_std

    return NormalDist().cdf(z_score)

In [123]:
clt_results = []

for n, exact in zip(session_lengths, exact_results):
    clt = clt_loss_probability(n)
    clt_results.append(clt)

    relative_error = abs(clt - exact) / exact * 100

    print(
        f"{n} games: exact = {exact:.8f}, "
        f"CLT = {clt:.8f}, "
        f"relative error = {relative_error:.2f}%"
    )

1 games: exact = 0.25000000, CLT = 0.24433531, relative error = 2.27%
5 games: exact = 0.18214380, CLT = 0.18107121, relative error = 0.59%
10 games: exact = 0.12318872, CLT = 0.11976764, relative error = 2.78%
20 games: exact = 0.06052126, CLT = 0.05663114, relative error = 6.43%
50 games: exact = 0.00898206, CLT = 0.00758058, relative error = 15.60%
100 games: exact = 0.00047273, CLT = 0.00033839, relative error = 28.42%


### Accuracy of the normal approximation

The continuity-corrected normal approximation underestimated the loss probability at every session length tested.

Relative error fell from 2.27% for one game to 0.59% for five games,
then increased to 28.42% for 100 games.

For 100 games, the exact loss probability was approximately 0.04727%,
compared with 0.03384% from the normal approximation.

As sessions get longer, a loss lies further into the tail of the
distribution. The CLT does not guarantee small relative errors for
these rare events, even as the distribution becomes more normal
near its centre.